In [1]:
!pip install pymongo

  Using cached pymongo-4.10.1-cp311-cp311-win_amd64.whl.metadata (22 kB)
  Using cached dnspython-2.7.0-py3-none-any.whl.metadata (5.8 kB)
Using cached pymongo-4.10.1-cp311-cp311-win_amd64.whl (876 kB)
Using cached dnspython-2.7.0-py3-none-any.whl (313 kB)


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pickle
from pymongo import MongoClient
from datetime import datetime

In [3]:
df = pd.read_csv('laptop_data.csv')

In [4]:
#Data preprocessing
# Drop unnecessary columns
df = df.drop(columns=['Unnamed: 0'])

In [5]:
# Clean and preprocess data
df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)  # Convert RAM to integer
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)  # Convert Weight to float

In [6]:
# One-hot encoding for categorical variables
df = pd.get_dummies(df, columns=['Company', 'TypeName', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys'], drop_first=True)

In [7]:
# Feature selection and target variable
Xfeatures = df.drop(columns=['Price'])  # Features (all columns except Price)
ylabels = df['Price']  # Target variable (Price)

In [8]:
# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(Xfeatures, ylabels, test_size=0.30, random_state=42)


In [9]:
# Initialize and train the model
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [10]:
# Evaluate the model
predictions = model.predict(x_test)
mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 281817566.79079753


In [11]:
# Test with a sample input
sample_input = x_test.iloc[0:1]
sample_prediction = model.predict(sample_input)
print(f"Sample Prediction: {sample_prediction}")

Sample Prediction: [68090.5659346]


In [12]:
# Serialize the model
model_binary = pickle.dumps(model)

In [13]:
# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")  # Connect to local MongoDB
db = client['LaptopPriceDB']  # Database
model_collection = db['models']  # Collection

In [14]:
# Store the model in MongoDB
model_document = {
    "model_name": "Linear_Regression_LaptopPrice",
    "framework": "scikit-learn",
    "model_binary": model_binary
}

In [15]:
model_collection.insert_one(model_document)
print("Model saved successfully in MongoDB!")

Model saved successfully in MongoDB!


In [16]:
# Store feature columns in MongoDB
features_binary = pickle.dumps(Xfeatures.columns)
model_collection.insert_one({"model_name": "LaptopPrice_Features", "features_binary": features_binary})

InsertOneResult(ObjectId('6778451d51677d1839a435d9'), acknowledged=True)

In [17]:
# Fetch the model from MongoDB
model_name = "Linear_Regression_LaptopPrice"
model_document = model_collection.find_one({"model_name": model_name})

In [18]:
if model_document:
    model_binary = model_document['model_binary']
    loaded_model = pickle.loads(model_binary)  # Deserialize the model
    print(f"Model '{model_name}' loaded successfully!")
else:
    print(f"Model '{model_name}' not found in the database.")

Model 'Linear_Regression_LaptopPrice' loaded successfully!


In [19]:
# Load feature columns
features_doc = model_collection.find_one({"model_name": "LaptopPrice_Features"})
if not features_doc:
    raise ValueError("Features not found in the database.")

loaded_features = pickle.loads(features_doc['features_binary'])
print("Feature columns loaded successfully.")

Feature columns loaded successfully.


In [20]:
# Insert dataset into MongoDB
collection = db["LaptopPrices"]
data = df.to_dict(orient="records")
collection.insert_many(data)
print("Dataset successfully saved to MongoDB!")

Dataset successfully saved to MongoDB!


In [21]:
# Fetch and print all documents from the model collection
for model_doc in model_collection.find():
    print(model_doc)

{'_id': ObjectId('6778450d51677d1839a435d8'), 'model_name': 'Linear_Regression_LaptopPrice', 'framework': 'scikit-learn', 'model_binary': b'\x80\x04\x95\xf4>\x00\x00\x00\x00\x00\x00\x8c\x1asklearn.linear_model._base\x94\x8c\x10LinearRegression\x94\x93\x94)\x81\x94}\x94(\x8c\rfit_intercept\x94\x88\x8c\x06copy_X\x94\x88\x8c\x06n_jobs\x94N\x8c\x08positive\x94\x89\x8c\x11feature_names_in_\x94\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01MQ\x01\x85\x94h\r\x8c\x05dtype\x94\x93\x94\x8c\x02O8\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01|\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK?t\x94b\x89]\x94(\x8c\x06Inches\x94\x8c\x03Ram\x94\x8c\x06Weight\x94\x8c\rCompany_Apple\x94\x8c\x0cCompany_Asus\x94\x8c\rCompany_Chuwi\x94\x8c\x0cCompany_Dell\x94\x8c\x0fCompany_Fujitsu\x94\x8c\x0eCompany_Google\x94\x8c\nCompany_HP\x94\x8c\x0eCompany_Huawei\x94\x8c\nCompany_LG\x94\x8c\x0eCompany_Lenovo\x94\x8c\x0bCompany_